# Model Application - CarT-Raji Dataset

This dataset consists of Raji B-cell cancer cells, treated with carT cells (for 4h / 24h).

Again here we use the Hotspot polarization & colocalization.

The modelling here shows clearly and intriguingly how different subsets of the carT conditions have different colocalization properties (especially w.r.t antigen-presenting phenotypes, markers like HLA-DR, CD54). A certain subset is well integrated with the control condition, while another subset clearly has significant colocalization properties, and an intermediate subset between the two. A plausible hypothesis is that we see the difference between the carT cells engaged with a cancer cell, and those that are not.

Observing the feature histograms, the model does slightly struggle with some bimodal features, notably CD4.

In [ ]:
import anndata
import pixelator
import torch
import scvi
import scipy
# from scvi import autotune

import seaborn as sns
import scanpy as sc
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
from tqdm import tqdm

from scib_metrics.benchmark import Benchmarker, BioConservation, BatchCorrection

# import ray
# from ray import tune

from pathlib import Path


from PixelGen.pxl_utils import train_model, get_model_latents, convert_polarization_to_feature_matrix, \
     convert_colocalization_to_feature_matrix, download_pxl
from PixelGen.scvi_utils import plot_losses, pca_neighbors_umap, calc_PCA, add_one_hot_encoding_obsm, plot_cumulative_variance
from PixelGen.common_utils import standardize, std_clip, filter_hv, split_pair_column, filter_df_by_two_columns, rank_plot
from PixelGen.metrics import MultiModalVIMetrics, distr_autocorrelation_in_latent
from PixelGen.multimodalvi import MultiModalSCVI
from PixelGen.multimodalvae import MultiModalVAE, AggMethod, D
from PixelGen.enums import AggMethod, D

# from pixelator.plot import molecule_rank_plot, cell_count_plot, scatter_umi_per_upia_vs_tau
# from pixelator.statistics import clr_transformation
# from pixelator.analysis.normalization import dsb_normalize


from sklearn.preprocessing import StandardScaler, MinMaxScaler 


import tempfile

from scvi import REGISTRY_KEYS
from scvi.module.base import (
    BaseModuleClass,
    LossOutput,
    PyroBaseModuleClass,
    auto_move_data,
)
from torch.distributions import NegativeBinomial, Normal, Poisson, MixtureSameFamily, Beta
from torch.distributions import kl_divergence as kl



# from cytovi import CytoVI

print(torch.cuda.is_available())


scvi.settings.seed = 0
print("Last run with scvi-tools version:", scvi.__version__)
sc.set_figure_params(figsize=(6, 6), frameon=False)
sns.set_theme()
torch.set_float32_matmul_precision("high")
CMAP = 'RdBu_r'
save_dir = tempfile.TemporaryDirectory()

%config InlineBackend.print_figure_kwargs={"facecolor": "w"}
%config InlineBackend.figure_format="retina"
%load_ext autoreload
%autoreload 2

In [ ]:
DATA_DIR = Path('./PixelGen/datasets/technote-cart-fmc63-v2.0')


FILENAMES = [
    "Sample01_human_pbmcs_control.layout.dataset.pxl",
    "Sample02_Raji_control.layout.dataset.pxl",
    "Sample03_CART_control.layout.dataset.pxl",
    "Sample04_CART_Raji_co-culture_4h.layout.dataset.pxl",
    "Sample05_CART_Raji_co-culture_24h.layout.dataset.pxl",
]

SAMPLE_NAMES = [
    "pbmc_control", 
    "raji_control",
    "carT_control",
    "carT_Raji_4h",
    "carT_Raji_24h",
]


COMBINED_FILENAME = "carT_combined.pxl"
COMBINED_PATH = DATA_DIR / COMBINED_FILENAME

# pg_data = pixelator.read(COMBINED_PATH)
# adata = pg_data.adata

# Uncomment to download for first time

# BASEURL = "https://pixelgen-technologies-datasets.s3.eu-north-1.amazonaws.com/mpx-datasets/pixelator/0.19.x/technote-cart-fmc63-v2.0"
# pg_data = download_pxl(
#     baseurl=BASEURL,
#     filenames=FILENAMES,
#     sample_names=SAMPLE_NAMES,
#     dataset_dir=DATA_DIR,
#     dataset_full_path=COMBINED_PATH,
# )

In [ ]:
# adata.write_h5ad(DATA_DIR / 'carT_hs_annotated_filtered.h5ad')
import sys
sys.path.append('/home/projects/nyosef/zvise/PixelGen/')

adata = anndata.read_h5ad(DATA_DIR / 'carT_hs_annotated_filtered.h5ad')

Preprocessing - at the end

## Feature Selection

In [ ]:
n_coloc_features = 100
calc_PCA(adata, rep='coloc_hs_c', key_added='coloc_hs_c')
coloc_autocorr = distr_autocorrelation_in_latent(adata, latent_keys=['coloc_hs_c_pca'], names=['coloc_hs_c_pca'],
                                                 rep_key='coloc_hs_c'
                                                 )
# blacklist = ['HLA-ABC', 'B2M']

coloc_autocorr = split_pair_column(coloc_autocorr.rename_axis('pair').reset_index(), c='pair')
# coloc_autocorr = filter_df_by_two_columns(coloc_autocorr, c1='marker_1', c2='marker_2', blacklist=blacklist)

coloc_autocorr.sort_values(by='morans', ascending=False, inplace=True)

ax = rank_plot(coloc_autocorr['morans'], s=5)
ax.axvline(x=n_coloc_features)
coloc_hvg_vars = coloc_autocorr['pair'][:n_coloc_features]

print(list(coloc_hvg_vars))

In [ ]:
n_pol_features = 40

calc_PCA(adata, rep='pol_hs_c', key_added='pol_hs_c')
pol_autocorr = distr_autocorrelation_in_latent(adata, latent_keys=['pol_hs_c_pca'], names=['pol_hs_c_pca'],
                                                 rep_key='pol_hs_c'
                                                 )
pol_hvg_vars = pol_autocorr.sort_values(by='morans', ascending=False).index[:n_pol_features]
print(list(pol_hvg_vars))

In [ ]:
# Instead of standardizing each feature separately, it worked better to choose a uniform scaling factor, based on 
# knowledge of the data, to bring the features  into unit values (when standardizing separately, 
# there is no difference in magnitude between less meaningful and more meaningful features)

adata.obsm['pol_hvg'] = adata.obsm['pol_hs_c'][pol_hvg_vars]
adata.obsm['coloc_hvg'] = adata.obsm['coloc_hs_c'][coloc_hvg_vars]

# adata.obsm['pol_hvg_std'] = std_clip(standardize(adata.obsm['pol_hvg']), c=3)
# adata.obsm['coloc_hvg_std'] = std_clip(standardize(adata.obsm['coloc_hvg']), c=3)

# adata.obsm['pol_std'] = std_clip(standardize(adata.obsm['pol_hs_c']), c=3)
# adata.obsm['coloc_std'] = std_clip(standardize(adata.obsm['coloc_hs_c']), c=3)

scale_factor = 10

adata.obsm['pol_hvg_std'] = scale_factor*adata.obsm['pol_hvg']
adata.obsm['coloc_hvg_std'] = scale_factor*adata.obsm['coloc_hvg']

adata.obsm['pol_std'] = scale_factor*adata.obsm['pol_hs_c']
adata.obsm['coloc_std'] = scale_factor*adata.obsm['coloc_hs_c']


print(f'Highly Variable Pol: {adata.obsm["pol_hvg"].shape[1]}/{adata.obsm["pol_hs_c"].shape[1]}')
print(f'Highly Variable Coloc: {adata.obsm["coloc_hvg"].shape[1]}/{adata.obsm["coloc_hs_c"].shape[1]}')

In [ ]:
ab_layer = 'clr'
pol_key = 'pol_hvg_std'
coloc_key = 'coloc_hvg_std'

In [ ]:
adata.obsm['ab_pol_coloc'] = pd.concat(
    (adata.to_df(ab_layer), adata.obsm[pol_key], adata.obsm[coloc_key]),
    axis=1,
)
calc_PCA(adata, rep='ab_pol_coloc', key_added='ab_pol_coloc')
calc_PCA(adata, rep=ab_layer, key_added=ab_layer)
calc_PCA(adata, rep=coloc_key, key_added=coloc_key)

In [ ]:
coloc_features = ['(CD54,HLA-DR)', ]
for f in coloc_features:
    adata.obs[f] = adata.obsm['coloc_hs_c'][f]

_ = pca_neighbors_umap(adata, ab_layer, umap_pl_kwargs=dict(layer=ab_layer, color=['sample', 'cell_type', *coloc_features], vcenter=0, cmap=CMAP), umap_title='Abundance Only')
_ = pca_neighbors_umap(adata, 'ab_pol_coloc', umap_pl_kwargs=dict(layer=ab_layer, color=['sample', 'cell_type', *coloc_features,], vcenter=0, cmap=CMAP), umap_title='Abundance + Pol + Coloc')
_ = pca_neighbors_umap(adata, coloc_key, umap_pl_kwargs=dict(layer=ab_layer, color=['sample', 'cell_type', *coloc_features,], vcenter=0, cmap=CMAP), umap_title='Coloc Only')

## Abundance Model

In [ ]:
model_cls = MultiModalSCVI
setup_kwargs = dict(layer=ab_layer, batch_key=None)
train_kwargs = dict(train_size=0.8, check_val_every_n_epoch=1, early_stopping=True, early_stopping_patience=200, batch_size=2000,
                    max_epochs=10000, enable_checkpointing=True, 
                    plan_kwargs=dict(lr=3e-4, optimizer='Adam', n_epochs_kl_warmup=400))
model_kwargs = dict(n_latent=20, n_hidden=128, n_layers=1, dropout_rate=0.1, distrs=[D.Normal,],)
modalities_latent_names=[(ab_layer, 'ab_model')]
abundance_model = train_model(adata, model_cls=model_cls, setup_kwargs=setup_kwargs, model_kwargs=model_kwargs, train_kwargs=train_kwargs,)
get_model_latents(adata, abundance_model, modalities_latent_names=modalities_latent_names)
for key, name in modalities_latent_names:
    title = name
    fig = pca_neighbors_umap(adata, name, umap_pl_kwargs=dict(color=['sample', 'cell_type', *coloc_features,], vcenter=0, cmap='RdBu_r', layer=ab_layer)).suptitle(title)

## Polarization Only

In [ ]:
pol_hvg_adata = anndata.AnnData(
    X=adata.obsm['pol_hvg_std'],
    obs=adata.obs,
    layers={
        'pol_hvg': adata.obsm['pol_hvg'],
        'pol_hvg_std': adata.obsm['pol_hvg_std'],
    }
)

In [ ]:
model_cls = MultiModalSCVI

setup_kwargs = dict(layer=pol_key, extra_modality_keys=[], n_modalities=1, batch_key=None, )
model_kwargs = dict(n_latent=20, n_hidden=128, n_layers=1, dropout_rate=0.1, 
                        distrs=[D.Normal], 
                        joint_kl=False, 
                        unimodal_kl=True,
                        external_kl_weight=1, 
                        decoder_kwargs=dict(decoder_param_eps=1e-2, decoder_activation='exp'),
                    )
train_kwargs = dict(train_size=0.8, check_val_every_n_epoch=1, early_stopping=True, 
                    early_stopping_patience=200, batch_size=2000,
                    max_epochs=10000, enable_checkpointing=True, 
                    plan_kwargs=dict(lr=1e-4, optimizer='Adam', n_epochs_kl_warmup=400))

latent_name = 'pol_model'
modalities_latent_names=[(pol_key, latent_name)]
pol_model = train_model(pol_hvg_adata, model_cls=model_cls, setup_kwargs=setup_kwargs, model_kwargs=model_kwargs, train_kwargs=train_kwargs)
get_model_latents(pol_hvg_adata, pol_model, modalities_latent_names=modalities_latent_names)
for key, name in modalities_latent_names:
    title = name
    fig = pca_neighbors_umap(pol_hvg_adata, name, umap_pl_kwargs=dict(color=['sample', 'cell_type', 'CD54', ], vcenter=0, cmap='RdBu_r', layer=pol_key)).suptitle(title)

## Colocalization Only

In [ ]:
obs = adata.obs.copy()
try:
    for key in coloc_features:
        obs.drop(columns=key, inplace=True)
except:
    pass

coloc_adata = anndata.AnnData(
    X=adata.obsm['coloc_std'],
    obs=obs,
    layers={
        'coloc_std': adata.obsm['coloc_std'],
        'coloc_c': adata.obsm['coloc_hs_c'],
    }
)

coloc_hvg_adata = anndata.AnnData(
    X=adata.obsm['coloc_hvg_std'],
    obs=obs,
    layers={
        'coloc_hvg': adata.obsm['coloc_hvg'],
        'coloc_hvg_std': adata.obsm['coloc_hvg_std'],
    }
)

In [ ]:
model_cls = MultiModalSCVI

cur_adata = coloc_hvg_adata
layer = 'coloc_hvg_std'

setup_kwargs = dict(layer=layer, extra_modality_keys=[], n_modalities=1, batch_key=None, )
model_kwargs = dict(n_latent=20, n_hidden=128, n_layers=2, dropout_rate=0.1, 
                        distrs=[D.Normal], 
                        joint_kl=False, unimodal_kl=True,
                        external_kl_weight=1, 
                        decoder_kwargs=dict(decoder_param_eps=1e-2, decoder_activation='exp'),
                    )
train_kwargs = dict(train_size=0.8, check_val_every_n_epoch=1, early_stopping=True, 
                    early_stopping_monitor='elbo_validation',
                    early_stopping_patience=200, batch_size=2000,
                    max_epochs=10000, enable_checkpointing=True, 
                    plan_kwargs=dict(lr=1e-4, optimizer='Adam', n_epochs_kl_warmup=400)
                    )

latent_name = 'coloc_model'
modalities_latent_names=[(layer, latent_name)]
coloc_model = train_model(cur_adata, model_cls=model_cls, setup_kwargs=setup_kwargs, model_kwargs=model_kwargs, train_kwargs=train_kwargs)
get_model_latents(cur_adata, coloc_model, modalities_latent_names=modalities_latent_names)
for key, name in modalities_latent_names:
    title = name
    fig = pca_neighbors_umap(cur_adata, name, 
        umap_pl_kwargs=dict(color=['sample', 'cell_type', *coloc_features], vcenter=0, cmap='RdBu_r', layer=layer)).suptitle(title)

## Shared Encoder

In [ ]:
model_cls = MultiModalSCVI

latent_name = 'shared_enc_model'

setup_kwargs = dict(layer=ab_layer, extra_modality_keys=[pol_key, coloc_key], n_modalities=3, batch_key=None, )
model_kwargs = dict(n_latent=30, n_hidden=128, n_layers=2, dropout_rate=0.1, 
                        distrs=[D.Normal, D.Normal, D.Normal], 
                        agg_method=AggMethod.SHARED_ENCODER,
                        loss_weights='auto',
                        joint_kl=True, unimodal_kl=False,
                        external_kl_weight=1,
                        decoder_kwargs=dict(decoder_param_eps=1e-2, decoder_activation='exp')
                    )
train_kwargs = dict(train_size=0.8, check_val_every_n_epoch=1, early_stopping=True, 
                    early_stopping_patience=200, batch_size=2000,
                    max_epochs=10000, enable_checkpointing=True, 
                    plan_kwargs=dict(lr=1e-4, optimizer='Adam', n_epochs_kl_warmup=400)
                )
shared_enc_model = train_model(adata, model_cls=model_cls, setup_kwargs=setup_kwargs, model_kwargs=model_kwargs, train_kwargs=train_kwargs,)

modalities_latent_names=[('joint', latent_name)]
get_model_latents(adata, shared_enc_model, modalities_latent_names=modalities_latent_names)
for key, name in modalities_latent_names:
    title = name
    fig = pca_neighbors_umap(adata, name, umap_pl_kwargs=dict(color=['sample', 'cell_type', *coloc_features,], 
                                                              vcenter=0, cmap='RdBu_r', layer=ab_layer)).suptitle(title)

## Metrics

In [ ]:
add_one_hot_encoding_obsm(adata, obs_column='cell_type')

In [ ]:
metrics = MultiModalVIMetrics(
    adata,
    models = {
        'abundance_only': abundance_model,
        # 'global_weights': learned_weights_model,
        'shared_enc': shared_enc_model,
        # 'shared_enc_w_batch': shared_enc_batch_key_model,
        'pol_model': pol_model,
        'coloc_model': coloc_model
    },
    pca_key='ab_pol_coloc_pca',
    additional_autocorr_keys=['cell_type'],
)
metrics.run()

In [ ]:
_ = metrics.mean_modality_errors_barplot()
_ = metrics.mean_modality_errors_barplot(reconstruction_mean=True)

In [ ]:
_ = metrics.mean_autocorr_barplot()
# _ = metrics.autocorr_barplot(autocorr_key=pol_key, auto_filter_features=10)
_ = metrics.autocorr_barplot(autocorr_key=coloc_key, features=coloc_hvg_vars[:10])
_ = metrics.autocorr_barplot(autocorr_key='cell_type', auto_filter_features=10, figsize=(10, 8))

In [ ]:
_ = metrics.feature_histplot(modality=coloc_key, features=coloc_hvg_vars[:3], hue='sample')
_ = metrics.feature_histplot(modality=ab_layer, features=['CD4', 'CD8', 'CD20',], hue='sample')

In [ ]:
_ = metrics.top_autocorr_features_barplot(key='coloc_hvg_std', model_names=['coloc_model', 'shared_enc', 'ab_pol_coloc_pca'], top=10)

## Preprocessing

In [ ]:
adata = pg_data.adata
adata.raw = adata.copy()
sc.pp.calculate_qc_metrics(adata, percent_top=None, inplace=True, var_type='proteins')
adata.layers['counts'] = adata.X.copy()

In [ ]:
sc.pl.violin(
    adata,
    ["n_proteins_by_counts", "total_counts",],
    groupby='sample',
    jitter=0.3,
    multi_panel=True,
)
cells_per_sample_df = (
    adata.obs.groupby("sample").size().to_frame(name="size").reset_index()
)

fig, ax = cell_count_plot(adata.obs, color_by="sample")

In [ ]:
molecule_rank_df = adata.obs[["sample", "molecules"]].copy()
molecule_rank_df["rank"] = molecule_rank_df.groupby(["sample"])["molecules"].rank(
    ascending=False, method="first"
)
fig_intersection, ax = molecule_rank_plot(molecule_rank_df, group_by="sample")
molecule_thresh = {
    'carT_Raji_4h': 10000,
    'carT_Raji_24h': 10000,
    'carT_control': 15000,
    'pbmc_control': 8000,
    'raji_control': 25000,
}
for sample, thresh in molecule_thresh.items():
    ax.axhline(thresh, color='black', linestyle='--')

In [ ]:
tau_metrics_df = adata.obs[["sample", "tau", "mean_molecules_per_a_pixel", "tau_type"]]
tau_metrics_df = tau_metrics_df.rename(columns={"mean_molecules_per_a_pixel": "umi_per_upia"})


fig, ax = scatter_umi_per_upia_vs_tau(tau_metrics_df, group_by="sample")

In [ ]:
import functools

components = adata.obs[
    (adata.obs['tau_type'] == 'normal') & 
    functools.reduce(lambda x,y: x | y, [((adata.obs['sample'] == sample) & (adata.obs['molecules'] > thresh)) for sample, thresh in molecule_thresh.items()])
].index
orig_adata = adata.copy()
adata = orig_adata[components, :]
cells_per_sample_df = (
    adata.obs.groupby("sample").size().to_frame(name="size").reset_index()
)

fig, ax = cell_count_plot(adata.obs, color_by="sample")
print(len(adata))

In [ ]:
fig, axes = plt.subplots(len(SAMPLE_NAMES), 2, figsize=(8, 4*len(SAMPLE_NAMES)))
for i, sample in enumerate(SAMPLE_NAMES):
    sample_adata = adata[adata.obs['sample'] == sample]
    ax = sc.pl.highest_expr_genes(sample_adata, n_top=20, show=False, ax=axes[i][0])
    ax.set_title(sample)
    stats = sample_adata.to_df().agg(['mean', 'var',], axis=0).T
    sns.scatterplot(x=stats['mean'], y=stats['var'], ax=axes[i][1])
    axes[i][1].loglog()
fig.tight_layout()

In [ ]:
isotype_controls=['mIgG1', 'mIgG2a', 'mIgG2b']
non_isotype_vars = [var for var in adata.var_names if var not in isotype_controls]

In [ ]:
adata.layers['dsb'] = dsb_normalize(adata.to_df('counts'), isotype_controls=isotype_controls)
adata.layers['clr'] = clr_transformation(adata.to_df('counts'), axis=1)
adata.layers['clr_by_ab'] = clr_transformation(adata.to_df('counts'), axis=0)
adata.layers['log1p'] = np.log1p(adata.to_df('counts'))

In [ ]:
adata = adata[adata.obs['sample'] != 'pbmc_control', :]

In [ ]:
sc.pp.pca(adata, layer='log1p')
sc.pp.neighbors(adata,)
sc.tl.umap(adata, )
sc.tl.leiden(adata)


# Old - for dsb data

# sc.tl.leiden(adata, restrict_to=('leiden', ['4']), resolution=0.5)
# cell_type_dict = {
#     '0': 'T_Cytotoxic', '2': 'T_Cytotoxic', '6': 'T_Cytotoxic',
#     '8': 'T_Helper',
#     '4,1': 'CD19_CD3_CD4_Doublets', '4,4': 'CD19_CD3_CD4_Doublets', 
#     **{f'4,{i}': 'T_Helper' for i in [0,2,3]},
#     '1': 'Raji',
#     '3': 'Raji',
#     '7': 'Raji',
#     '9': 'CD19_CD3_CD8_Doublets',
#     '10': 'T_DN',
#     '11': 'CD19_CD3_Doublets',
#     '5': 'CD19_CD3_CD8_Doublets',
# }

# New - log1p data
# cell_type_dict = {
#     '2': 'Raji', '3': 'Raji', '4': 'Raji',
#     '9': 'CD19_CD3_CD8_Doublets', '7': 'CD19_CD3_CD8_Doublets',
#     '0': 'T_Cytotoxic', '1': 'T_Cytotoxic', '5': 'T_Cytotoxic',
#     '6': 'T_Helper', '8': 'T_Helper',
#     '10': 'T_DN',
#     '11,0': 'Raji', '11,1': 'CD19_CD3_CD8_Doublets',
# }

# adata.obs['cell_type'] = adata.obs['leiden_R'].map(cell_type_dict)
# sc.pl.umap(adata, layer='log1p', color=['sample', 'cell_type', 'leiden_R', 'CD19', 'FMC63', 'CD3E', 'CD4', 'CD8',])

sc.tl.leiden(adata, restrict_to=('leiden', ['12']), resolution=0.5)
cell_type_dict = {
    '0': 'Raji', '4': 'Raji', '12,0': 'Raji',
    '9': 'Doublets', '6': 'Doublets', '12,1': 'Doublets', '12,2': 'Doublets',
    '1': 'T_Cytotoxic', '2': 'T_Cytotoxic', '3': 'T_Cytotoxic', '7': 'T_Cytotoxic', 
    '5': 'T_Helper', '8': 'T_Helper',
    '11': 'T_DN', '9': 'B_doublets', '10': 'B_doublets'
}
adata.obs['cell_type'] = adata.obs['leiden_R'].map(cell_type_dict)
sc.pl.umap(adata, layer='log1p', color=['sample', 'leiden_R', 'cell_type', 'log1p_total_counts', 'CD19', 'CD20', 'FMC63', 'CD3E', 'CD4', 'CD8',])


In [ ]:
sc.tl.rank_genes_groups(adata, groupby='leiden_R', groups=['10', '9'], layer='log1p', use_raw=False, method='wilcoxon')
sc.pl.rank_genes_groups(adata)



# pd.set_option('display.max_rows', None)
# all_de = sc.get.rank_genes_groups_df(adata, group=None, pval_cutoff=0.05)
# pos_de = all_de[all_de['logfoldchanges'] > 0]
# neg_de = all_de[all_de['logfoldchanges'] < 0]

# for group in '9', '10':
#     print(pos_de[pos_de['group'] == group].sort_values(by='pvals', ascending=True).head(5))
#     print(neg_de[neg_de['group'] == group].sort_values(by='pvals', ascending=True).head(5))

# .sort_values(by='pvals', ascending=True).head(200)
# sc.pl.rank_genes_groups_dotplot(adata, n_genes=4, groups=['10', '9'], groupby='leiden_R', values_to_plot="logfoldchanges", vcenter=0, cmap='coolwarm')

In [ ]:
adata.obsm['pol_hs_c'] = adata.uns['pol_hs'].pivot_table(
            index='component', columns='marker', values='pol_hs', fill_value=0, observed=True).reindex(adata.obs.index)

adata.obsm['coloc_hs_c'] = adata.uns['coloc_hs'].pivot_table(
            index='component', columns='pair_name', values='coloc_hs', fill_value=0, observed=True).reindex(adata.obs.index)